# 02 - Data Cleaning

**Tasks**:
- Standardize column names
- Convert data types
- Handle missing values
- Remove duplicates
- Validate data quality

**Output**: Cleaned data saved to `data/processed/`

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)

# S3 paths
BASE = "https://validex-ml-data.s3.us-east-1.amazonaws.com"

# Local output paths
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

In [5]:
# Load raw data
daily = pd.read_parquet(f"{BASE}/daily_adjusted/ALL_daily_adjusted.parquet")
income = pd.read_parquet(f"{BASE}/fundamentals/ALL_income_statement.parquet")
balance = pd.read_parquet(f"{BASE}/fundamentals/ALL_balance_sheet.parquet")
cashflow = pd.read_parquet(f"{BASE}/fundamentals/ALL_cash_flow.parquet")
options = pd.read_parquet(f"{BASE}/options/ALL_options.parquet")
overview = pd.read_csv(f"{BASE}/fundamentals/ALL_overview.csv")

print("Data loaded successfully!")

Data loaded successfully!


## 1. Clean Daily Prices

In [6]:
print(f"Shape: {daily.shape}")
print(f"\nColumns: {daily.columns.tolist()}")
print(f"\nData types:\n{daily.dtypes}")
print(f"\nMissing values:\n{daily.isnull().sum()}")
print(f"\nSample:")
daily.head()

Shape: (27516, 10)

Columns: ['date', 'symbol', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'dividend', 'split_coeff']

Data types:
date           datetime64[ns]
symbol                 object
open                  float64
high                  float64
low                   float64
close                 float64
adj_close             float64
volume                  int64
dividend              float64
split_coeff           float64
dtype: object

Missing values:
date           0
symbol         0
open           0
high           0
low            0
close          0
adj_close      0
volume         0
dividend       0
split_coeff    0
dtype: int64

Sample:


,date,symbol,open,high,low,close,adj_close,volume,dividend,split_coeff
0,2015-01-02,ADMA,11.2500,11.5000,11.2000,11.4500,11.4500,3300,0.0,1.0
1,2015-01-05,ADMA,11.5000,11.5190,11.1100,11.1100,11.1100,1600,0.0,1.0
2,2015-01-06,ADMA,11.0400,11.5000,11.0400,11.2200,11.2200,2800,0.0,1.0
3,2015-01-07,ADMA,11.0600,11.9499,10.9100,11.0201,11.0201,5600,0.0,1.0
4,2015-01-08,ADMA,10.9101,11.5000,10.9101,11.0200,11.0200,3100,0.0,1.0


In [7]:
def clean_daily(df):
    df = df.copy()
    
    # Standardize column names
    df.columns = df.columns.str.lower().str.strip()
    
    # Rename for consistency
    df = df.rename(columns={
        'adj_close': 'adjusted_close',
        'split_coeff': 'split_coefficient'
    })
    
    # Convert date
    df['date'] = pd.to_datetime(df['date'])
    
    # Ensure numeric columns
    numeric_cols = ['open', 'high', 'low', 'close', 'adjusted_close', 'volume', 'dividend', 'split_coefficient']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Remove duplicates
    df = df.drop_duplicates(subset=['symbol', 'date'])
    
    # Sort
    df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
    
    return df

daily_clean = clean_daily(daily)
print(f"Cleaned shape: {daily_clean.shape}")
print(f"Date range: {daily_clean['date'].min()} to {daily_clean['date'].max()}")
print(f"Tickers: {daily_clean['symbol'].unique()}")

Cleaned shape: (27516, 10)
Date range: 2015-01-02 00:00:00 to 2025-12-31 00:00:00
Tickers: ['AAPL' 'ADMA' 'AMZN' 'AXON' 'GOOG' 'META' 'MSFT' 'NTRA' 'NVDA' 'SHAK']


In [8]:
# Check for missing values after cleaning
print("Missing values:")
print(daily_clean.isnull().sum())

# Check rows per ticker
print("\nRows per ticker:")
print(daily_clean.groupby('symbol').size())

Missing values:
date                 0
symbol               0
open                 0
high                 0
low                  0
close                0
adjusted_close       0
volume               0
dividend             0
split_coefficient    0
dtype: int64

Rows per ticker:
symbol
AAPL    2766
ADMA    2766
AMZN    2766
AXON    2766
GOOG    2766
META    2766
MSFT    2766
NTRA    2641
NVDA    2766
SHAK    2747
dtype: int64


## 2. Clean Income Statement

In [9]:
print(f"Shape: {income.shape}")
print(f"\nColumns: {income.columns.tolist()}")
print(f"\nMissing values:\n{income.isnull().sum()}")

Shape: (736, 27)

Columns: ['symbol', 'fiscalDateEnding', 'reportedCurrency', 'grossProfit', 'totalRevenue', 'costOfRevenue', 'costofGoodsAndServicesSold', 'operatingIncome', 'sellingGeneralAndAdministrative', 'researchAndDevelopment', 'operatingExpenses', 'investmentIncomeNet', 'netInterestIncome', 'interestIncome', 'interestExpense', 'nonInterestIncome', 'otherNonOperatingIncome', 'depreciation', 'depreciationAndAmortization', 'incomeBeforeTax', 'incomeTaxExpense', 'interestAndDebtExpense', 'netIncomeFromContinuingOperations', 'comprehensiveIncomeNetOfTax', 'ebit', 'ebitda', 'netIncome']

Missing values:
symbol                                 0
fiscalDateEnding                       0
reportedCurrency                       0
grossProfit                           17
totalRevenue                          20
costOfRevenue                         17
costofGoodsAndServicesSold            17
operatingIncome                        0
sellingGeneralAndAdministrative        9
researchAndDevelo

In [10]:
def clean_fundamentals(df, date_col='fiscalDateEnding'):
    df = df.copy()
    
    # Standardize column names
    df.columns = df.columns.str.strip()
    
    # Standardize symbol column
    if 'Symbol' in df.columns:
        df = df.rename(columns={'Symbol': 'symbol'})
    
    # Convert date
    df[date_col] = pd.to_datetime(df[date_col])
    
    # Convert numeric columns (exclude symbol, date, currency)
    exclude_cols = ['symbol', date_col, 'reportedCurrency']
    numeric_cols = [col for col in df.columns if col not in exclude_cols]
    
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Remove duplicates
    df = df.drop_duplicates(subset=['symbol', date_col])
    
    # Sort
    df = df.sort_values(['symbol', date_col]).reset_index(drop=True)
    
    return df

income_clean = clean_fundamentals(income)
print(f"Cleaned shape: {income_clean.shape}")
print(f"Date range: {income_clean['fiscalDateEnding'].min()} to {income_clean['fiscalDateEnding'].max()}")

Cleaned shape: (736, 27)
Date range: 2005-12-31 00:00:00 to 2026-01-31 00:00:00


## 3. Clean Balance Sheet

In [11]:
print(f"Shape: {balance.shape}")
print(f"\nMissing values (top 10):")
print(balance.isnull().sum().sort_values(ascending=False).head(10))

Shape: (729, 39)

Missing values (top 10):
otherNonCurrentAssets                     729
longTermDebtNoncurrent                    729
investments                               729
deferredRevenue                           729
currentDebt                               729
accumulatedDepreciationAmortizationPPE    725
currentLongTermDebt                       508
longTermInvestments                       448
capitalLeaseObligations                   394
treasuryStock                             368
dtype: int64


In [12]:
balance_clean = clean_fundamentals(balance)
print(f"Cleaned shape: {balance_clean.shape}")
print(f"Date range: {balance_clean['fiscalDateEnding'].min()} to {balance_clean['fiscalDateEnding'].max()}")

Cleaned shape: (729, 39)
Date range: 2005-12-31 00:00:00 to 2026-01-31 00:00:00


## 4. Clean Cash Flow

In [13]:
print(f"Shape: {cashflow.shape}")
print(f"\nMissing values (top 10):")
print(cashflow.isnull().sum().sort_values(ascending=False).head(10))

Shape: (731, 30)

Missing values (top 10):
proceedsFromRepaymentsOfShortTermDebt                        731
proceedsFromSaleOfTreasuryStock                              731
proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet    731
proceedsFromIssuanceOfCommonStock                            731
dividendPayoutPreferredStock                                 731
paymentsForRepurchaseOfPreferredStock                        731
paymentsForRepurchaseOfEquity                                731
paymentsForRepurchaseOfCommonStock                           731
profitLoss                                                   731
changeInOperatingAssets                                      731
dtype: int64


In [14]:
cashflow_clean = clean_fundamentals(cashflow)
print(f"Cleaned shape: {cashflow_clean.shape}")
print(f"Date range: {cashflow_clean['fiscalDateEnding'].min()} to {cashflow_clean['fiscalDateEnding'].max()}")

Cleaned shape: (731, 30)
Date range: 2005-12-31 00:00:00 to 2026-01-31 00:00:00


## 5. Clean Options

In [15]:
print(f"Shape: {options.shape}")
print(f"\nColumns: {options.columns.tolist()}")
print(f"\nMissing values:\n{options.isnull().sum()}")
print(f"\nSample:")
options.head()

Shape: (1854022, 20)

Columns: ['contractID', 'symbol', 'expiration', 'strike', 'call_put', 'last', 'mark', 'bid', 'bid_size', 'ask', 'ask_size', 'volume', 'open_interest', 'trade_date', 'implied_vol', 'delta', 'gamma', 'theta', 'vega', 'rho']

Missing values:
contractID       0
symbol           0
expiration       0
strike           0
call_put         0
last             0
mark             0
bid              0
bid_size         0
ask              0
ask_size         0
volume           0
open_interest    0
trade_date       0
implied_vol      0
delta            0
gamma            0
theta            0
vega             0
rho              0
dtype: int64

Sample:


,contractID,symbol,expiration,strike,call_put,last,mark,bid,bid_size,ask,ask_size,volume,open_interest,trade_date,implied_vol,delta,gamma,theta,vega,rho
0,ADMA180817C00002500,ADMA,2018-08-17,2.5,call,0.00,3.95,2.6,5,5.30,5,0,0,2018-08-01,2.74652,0.97280,0.01704,-0.00733,0.00084,0.00100
1,ADMA180817P00002500,ADMA,2018-08-17,2.5,put,0.00,0.01,0.0,0,1.30,15,0,0,2018-08-01,2.10263,-0.00920,0.00880,-0.00218,0.00033,-0.00003
2,ADMA180817C00005000,ADMA,2018-08-17,5.0,call,0.75,1.20,0.2,50,2.20,2,0,10,2018-08-01,0.01488,1.00000,0.00000,-0.00026,0.00000,0.00219
3,ADMA180817P00005000,ADMA,2018-08-17,5.0,put,0.18,0.01,0.0,0,0.85,15,0,14,2018-08-01,0.64901,-0.02932,0.07678,-0.00180,0.00089,-0.00009
4,ADMA180817C00007500,ADMA,2018-08-17,7.5,call,0.13,0.01,0.0,0,1.00,15,0,41,2018-08-01,0.43438,0.04558,0.16455,-0.00176,0.00128,0.00012


In [16]:
def clean_options(df):
    df = df.copy()
    
    # Standardize column names
    df.columns = df.columns.str.lower().str.strip()
    
    # Convert dates
    df['expiration'] = pd.to_datetime(df['expiration'])
    
    # Check for trade_date column
    if 'trade_date' in df.columns:
        df['trade_date'] = pd.to_datetime(df['trade_date'])
    
    # Ensure numeric columns
    numeric_cols = ['strike', 'last', 'mark', 'bid', 'ask', 'bid_size', 'ask_size', 
                    'volume', 'open_interest', 'implied_volatility', 
                    'delta', 'gamma', 'theta', 'vega', 'rho']
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Standardize call_put column
    if 'call_put' in df.columns:
        df['call_put'] = df['call_put'].str.upper().str.strip()
    
    # Remove duplicates
    df = df.drop_duplicates(subset=['contractid']) if 'contractid' in df.columns else df.drop_duplicates()
    
    # Sort
    sort_cols = ['symbol', 'trade_date', 'expiration', 'strike'] if 'trade_date' in df.columns else ['symbol', 'expiration', 'strike']
    df = df.sort_values(sort_cols).reset_index(drop=True)
    
    return df

options_clean = clean_options(options)
print(f"Cleaned shape: {options_clean.shape}")
print(f"Expiration range: {options_clean['expiration'].min()} to {options_clean['expiration'].max()}")
print(f"Call/Put split:\n{options_clean['call_put'].value_counts()}")

Cleaned shape: (657414, 20)
Expiration range: 2015-02-06 00:00:00 to 2028-06-16 00:00:00
Call/Put split:
call_put
CALL    328707
PUT     328707
Name: count, dtype: int64


## 6. Clean Overview

In [17]:
print(f"Shape: {overview.shape}")
print(f"\nColumns: {overview.columns.tolist()}")
overview

Shape: (10, 55)

Columns: ['Symbol', 'AssetType', 'Name', 'Description', 'CIK', 'Exchange', 'Currency', 'Country', 'Sector', 'Industry', 'Address', 'OfficialSite', 'FiscalYearEnd', 'LatestQuarter', 'MarketCapitalization', 'EBITDA', 'PERatio', 'PEGRatio', 'BookValue', 'DividendPerShare', 'DividendYield', 'EPS', 'RevenuePerShareTTM', 'ProfitMargin', 'OperatingMarginTTM', 'ReturnOnAssetsTTM', 'ReturnOnEquityTTM', 'RevenueTTM', 'GrossProfitTTM', 'DilutedEPSTTM', 'QuarterlyEarningsGrowthYOY', 'QuarterlyRevenueGrowthYOY', 'AnalystTargetPrice', 'AnalystRatingStrongBuy', 'AnalystRatingBuy', 'AnalystRatingHold', 'AnalystRatingSell', 'AnalystRatingStrongSell', 'TrailingPE', 'ForwardPE', 'PriceToSalesRatioTTM', 'PriceToBookRatio', 'EVToRevenue', 'EVToEBITDA', 'Beta', '52WeekHigh', '52WeekLow', '50DayMovingAverage', '200DayMovingAverage', 'SharesOutstanding', 'SharesFloat', 'PercentInsiders', 'PercentInstitutions', 'DividendDate', 'ExDividendDate']


,Symbol,AssetType,Name,Description,CIK,Exchange,Currency,Country,Sector,Industry,Address,OfficialSite,FiscalYearEnd,LatestQuarter,MarketCapitalization,EBITDA,PERatio,PEGRatio,BookValue,DividendPerShare,DividendYield,EPS,RevenuePerShareTTM,ProfitMargin,OperatingMarginTTM,...,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,AnalystTargetPrice,AnalystRatingStrongBuy,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongSell,TrailingPE,ForwardPE,PriceToSalesRatioTTM,PriceToBookRatio,EVToRevenue,EVToEBITDA,Beta,52WeekHigh,52WeekLow,50DayMovingAverage,200DayMovingAverage,SharesOutstanding,SharesFloat,PercentInsiders,PercentInstitutions,DividendDate,ExDividendDate
0,ADMA,Common Stock,ADMA Biologics Inc,"ADMA Biologics, Inc., a biopharmaceutical comp...",1368514,NASDAQ,USD,USA,HEALTHCARE,BIOTECHNOLOGY,"465 STATE ROUTE 17, RAMSEY, NJ, UNITED STATES,...",https://www.admabiologics.com,December,2025-12-31,3746502000,199539000,18.26,0.000,2.007,NaN,NaN,0.86,2.141,0.2880,0.4510,...,-0.528,0.184,25.67,0,3,0,0,0,18.26,28.99,7.340,9.13,8.100,23.00,0.593,25.67,13.76,17.22,17.37,237998000,233296000,2.723,94.696,NaN,NaN
1,NTRA,Common Stock,Natera Inc,"Natera, Inc., a diagnostic company, develops a...",1604821,NASDAQ,USD,USA,HEALTHCARE,DIAGNOSTICS & RESEARCH,"13011 MCCALLEN PASS, AUSTIN, TX, UNITED STATES...",https://www.natera.com,December,2025-12-31,27940899000,-271427008,NaN,0.000,12.260,NaN,NaN,-1.52,16.870,-0.0903,-0.0342,...,0.000,0.398,260.65,5,13,2,0,0,-,-,12.120,22.61,12.980,-26.80,1.702,256.36,125.38,225.32,188.81,141731000,137472000,3.155,94.606,NaN,NaN
2,AXON,Common Stock,Axon Enterprise Inc.,"Axon Enterprise, Inc. develops, manufactures, ...",1069183,NASDAQ,USD,USA,INDUSTRIALS,AEROSPACE & DEFENSE,"17800 NORTH 85TH STREET, SCOTTSDALE, AZ, UNITE...",https://www.axon.com,December,2025-12-31,45866066000,53508000,385.47,1.645,40.430,NaN,NaN,1.48,35.600,0.0449,-0.0306,...,-0.980,0.385,735.01,8,10,1,0,0,385.47,72.46,16.500,14.13,16.550,234.18,1.524,885.92,396.41,539.25,667.42,80398000,76381000,4.385,85.899,NaN,NaN
3,SHAK,Common Stock,Shake Shack Inc,"Shake Shack Inc. owns, operates and licenses S...",1620533,NYSE,USD,USA,CONSUMER CYCLICAL,RESTAURANTS,"225 VARICK STREET, NEW YORK, NY, UNITED STATES...",https://www.shakeshack.com,December,2025-12-31,4132208000,174356000,88.80,2.550,13.050,NaN,NaN,1.09,35.940,0.0316,0.0513,...,0.287,0.219,113.30,2,11,12,0,1,88.8,65.36,2.859,7.46,3.088,24.31,1.766,144.65,72.93,91.56,103.62,40257700,38455400,4.784,107.741,NaN,NaN
4,AAPL,Common Stock,Apple Inc,Apple Inc. is an American multinational techno...,320193,NASDAQ,USD,USA,TECHNOLOGY,CONSUMER ELECTRONICS,"ONE APPLE PARK WAY, CUPERTINO, CA, UNITED STAT...",https://www.apple.com,September,2025-12-31,3825723245000,152901992000,33.24,2.331,6.000,1.03,0.0039,7.83,29.300,0.2700,0.3540,...,0.183,0.157,292.15,5,25,16,1,1,33.24,30.49,8.780,43.33,8.830,25.15,1.116,288.35,168.48,264.56,243.93,14681140000,14656182000,1.637,65.200,2026-02-12,2026-02-09
5,MSFT,Common Stock,Microsoft Corporation,Microsoft Corporation is an American multinati...,789019,NASDAQ,USD,USA,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,"ONE MICROSOFT WAY, REDMOND, WA, UNITED STATES,...",https://www.microsoft.com,June,2025-12-31,3052328976000,175258993000,25.38,1.340,52.620,3.48,0.0086,16.18,41.100,0.3900,0.4710,...,0.598,0.167,596.00,10,44,3,0,0,25.38,21.51,9.990,7.80,9.880,16.03,1.108,552.24,342.17,439.27,484.80,7425629000,7414862000,0.079,75.891,2026-03-12,2026-02-19
6,NVDA,Common Stock,NVIDIA Corporation,Nvidia Corporation is an American multinationa...,1045810,NASDAQ,USD,USA,TECHNOLOGY,SEMICONDUCTORS,"2788 SAN TOMAS EXPRESSWAY, SANTA CLARA, CA, UN...",https://www.nvidia.com,January,2026-01-31,4456078901000,133230002000,37.42,1.113,6.470,0.04,0.0002,4.90,8.870,0.5560,0.6500,...,0.956,0.732,265.18,11,47,2,1,0,37.42,22.94,20.640,28.32,20.390,30.46,2.375,212.18,86.60,186.12,175.92,24300000000,23330673000,4.205,69.690,2025-12-26,2026-03-11
7,AMZN,Common Stock,Amazon.com Inc,"Amazon.c

In [18]:
def clean_overview(df):
    df = df.copy()
    
    # Standardize symbol column
    df = df.rename(columns={'Symbol': 'symbol'})
    
    # Select relevant columns for our project
    keep_cols = [
        'symbol', 'Name', 'Sector', 'Industry', 'Exchange',
        'SharesOutstanding', 'DividendYield', 'Beta'
    ]
    
    # Keep only columns that exist
    keep_cols = [col for col in keep_cols if col in df.columns]
    df = df[keep_cols]
    
    # Standardize column names
    df.columns = df.columns.str.lower()
    
    # Convert numeric columns
    numeric_cols = ['sharesoutstanding', 'dividendyield', 'beta']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

overview_clean = clean_overview(overview)
print(f"Cleaned shape: {overview_clean.shape}")
overview_clean

Cleaned shape: (10, 8)


,symbol,name,sector,industry,exchange,sharesoutstanding,dividendyield,beta
0,ADMA,ADMA Biologics Inc,HEALTHCARE,BIOTECHNOLOGY,NASDAQ,237998000,NaN,0.593
1,NTRA,Natera Inc,HEALTHCARE,DIAGNOSTICS & RESEARCH,NASDAQ,141731000,NaN,1.702
2,AXON,Axon Enterprise Inc.,INDUSTRIALS,AEROSPACE & DEFENSE,NASDAQ,80398000,NaN,1.524
3,SHAK,Shake Shack Inc,CONSUMER CYCLICAL,RESTAURANTS,NYSE,40257700,NaN,1.766
4,AAPL,Apple Inc,TECHNOLOGY,CONSUMER ELECTRONICS,NASDAQ,14681140000,0.0039,1.116
5,MSFT,Microsoft Corporation,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,NASDAQ,7425629000,0.0086,1.108
6,NVDA,NVIDIA Corporation,TECHNOLOGY,SEMICONDUCTORS,NASDAQ,24300000000,0.0002,2.375
7,AMZN,Amazon.com Inc,CONSUMER CYCLICAL,INTERNET RETAIL,NASDAQ,10734921000,NaN,1.420
8,GOOG,Alphabet Inc Class C,COMMUNICATION SERVICES,INTERNET CONTENT & INFORMATION,NASDAQ,5438000000,0.0027,1.112
9,META,Meta Platforms Inc.,COMMUNICATION SERVICES,INTERNET CONTENT & INFORMATION,NASDAQ,2187178000,0.0031,1.279


## 7. Data Quality Summary

In [19]:
datasets = {
    'daily': daily_clean,
    'income': income_clean,
    'balance': balance_clean,
    'cashflow': cashflow_clean,
    'options': options_clean,
    'overview': overview_clean
}

print("----- Data Quality Summary -----")
for name, df in datasets.items():
    missing_pct = (df.isnull().sum().sum() / df.size) * 100
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Missing: {missing_pct:.2f}%")
    print(f"  Tickers: {df['symbol'].nunique()}")

----- Data Quality Summary -----

daily:
  Shape: (27516, 10)
  Missing: 0.00%
  Tickers: 10

income:
  Shape: (736, 27)
  Missing: 26.68%
  Tickers: 10

balance:
  Shape: (729, 39)
  Missing: 26.90%
  Tickers: 10

cashflow:
  Shape: (731, 30)
  Missing: 56.06%
  Tickers: 10

options:
  Shape: (657414, 20)
  Missing: 0.00%
  Tickers: 10

overview:
  Shape: (10, 8)
  Missing: 6.25%
  Tickers: 10


## 8. Save Cleaned Data

In [20]:
# Save to processed folder
daily_clean.to_parquet(PROCESSED / 'daily_clean.parquet', index=False)
income_clean.to_parquet(PROCESSED / 'income_clean.parquet', index=False)
balance_clean.to_parquet(PROCESSED / 'balance_clean.parquet', index=False)
cashflow_clean.to_parquet(PROCESSED / 'cashflow_clean.parquet', index=False)
options_clean.to_parquet(PROCESSED / 'options_clean.parquet', index=False)
overview_clean.to_parquet(PROCESSED / 'overview_clean.parquet', index=False)

print("Cleaned data saved to:")
for f in PROCESSED.glob('*.parquet'):
    print(f"  {f}")

Cleaned data saved to:
  ../data/processed/options_clean.parquet
  ../data/processed/cashflow_clean.parquet
  ../data/processed/balance_clean.parquet
  ../data/processed/overview_clean.parquet
  ../data/processed/daily_clean.parquet
  ../data/processed/income_clean.parquet
